# Weapon Detector — Colab Training

Fine-tunes `yolo11s.pt` on merged public weapon datasets for the GP-Model security pipeline. Runs end-to-end on a free T4 runtime (~1h for 50 epochs on a ~5k-image merge).

**Runtime → Change runtime type → GPU (T4 / A100).**

Workflow:
1. Verify GPU, clone the repo, install deps
2. Upload (or download) your datasets
3. Merge them into one YOLO-format dataset
4. Train
5. Evaluate
6. Export + download the weights

## 1. GPU + repo + deps

In [ ]:
!nvidia-smi | head -n 20

In [ ]:
# Install uv and clone the repo.
!pip install -q uv
!git clone https://github.com/Turki-Alrowithi/GP-Model.git gpmodel
%cd gpmodel

In [ ]:
# Install project deps (torch + ultralytics + sahi + supervision).
# Takes ~3-4 minutes on a fresh Colab runtime.
!uv sync

## 2. Get the source datasets

Option A — upload a zip of each source dataset to `/content/datasets_raw/`. Option B — pull them from Roboflow / Kaggle. Whatever you do, each source should end up with:

```
/content/datasets_raw/<name>/
├── images/{train,val,test}/
├── labels/{train,val,test}/
└── data.yaml             # Ultralytics-format
```

In [ ]:
from google.colab import files
# Uncomment to upload zips; comment out if you're scripting downloads instead.
# files.upload()

In [ ]:
# Example: unzip + link each source into our datasets/ folder.
!mkdir -p datasets_raw
# !unzip -q sohas.zip -d datasets_raw/sohas
# !unzip -q olmos.zip -d datasets_raw/olmos

# Place a copy of each source's data.yaml under datasets/ so the
# merger can find it. Adjust paths as needed.
# !cp datasets_raw/sohas/data.yaml datasets/sohas.yaml
# !cp datasets_raw/olmos/data.yaml datasets/olmos.yaml

## 3. Merge into a weapon vocabulary

Copy the template and edit it for your actual source class names:

In [ ]:
!cp datasets/class_map.example.yaml datasets/class_map.yaml
!cat datasets/class_map.yaml

In [ ]:
!uv run python apps/merge_datasets.py \
    --sources datasets/sohas.yaml datasets/olmos.yaml \
    --mapping datasets/class_map.yaml \
    --output datasets/weapons_merged \
    --copy

## 4. Train

On a T4, start with a shorter run to sanity-check (~5 epochs, ~5 minutes). Once you're happy with the loss curve, push to 50-100 epochs.

In [ ]:
!uv run python apps/train.py \
    --weights yolo11s.pt \
    --data datasets/weapons_merged/data.yaml \
    --epochs 50 \
    --imgsz 640 \
    --batch 32 \
    --device 0 \
    --name weapons

## 5. Evaluate

Reports mAP@50, mAP@50-95, precision, recall on the validation split.

In [ ]:
!uv run python apps/eval.py \
    --weights runs/train/weapons/weights/best.pt \
    --data datasets/weapons_merged/data.yaml \
    --device 0

## 6. Export and download weights

Exports to ONNX (portable) — CoreML is currently blocked by an upstream coremltools/torch incompat and can be re-tried on the host machine once fixed.

In [ ]:
!uv run python apps/export.py export \
    --weights runs/train/weapons/weights/best.pt \
    --format onnx \
    --imgsz 640

In [ ]:
from google.colab import files
files.download('runs/train/weapons/weights/best.pt')
files.download('models/best.onnx')

## 7. Deploy back to the laptop

Drop the downloaded weights into `models/weapons.pt` (or `.onnx`) and point a detector config at them:

```yaml
detector:
  type: sahi_yolo        # recommended for small-object weapons
  weights: models/weapons.pt
  device: mps
  slice_height: 640
  slice_width: 640
  overlap_height_ratio: 0.20
  overlap_width_ratio: 0.20
```

Then enable the weapon rule:

```yaml
rules:
  weapon:
    enabled: true
    classes: [firearm, knife]
    min_confidence: 0.60
    min_consecutive_frames: 3
    severity: CRITICAL
    cooldown_s: 30
```